# 4 · Query Expansion (Claude)

Expands the 15 benchmark questions into formal legal terminology using **Claude via LangChain**, with a JSON checkpoint so an interrupted run resumes.

Output: `data/processed_data/benchmarking_data_expanded.csv`.

In [1]:
import json, time
import pandas as pd
from config import config
from indian_marriage_legal_recommender.query_expansion import expand_query

df = pd.read_csv(config.BENCHMARK_CSV)
ckpt = config.PROCESSED_DIR / 'expansion_checkpoint.json'
cache = json.loads(ckpt.read_text()) if ckpt.exists() else {}
print(f'{len(df)} questions · {len(cache)} already expanded')

15 questions · 0 already expanded


### The expansion prompt

`expand_query` runs a LangChain LCEL chain (`prompt | Claude | StrOutputParser`). The system + human prompt below define the guardrails — preserve intent, add legal terminology, never answer or broaden the query.

In [2]:
from indian_marriage_legal_recommender import query_expansion

print('SYSTEM PROMPT\n' + '-' * 60)
print(query_expansion._SYSTEM)
print('\nHUMAN PROMPT\n' + '-' * 60)
print(query_expansion._HUMAN)

SYSTEM PROMPT
------------------------------------------------------------
You are an Indian Legal Query Expansion Assistant. You rewrite queries using generic legal vocabulary ONLY. You have no knowledge of specific statutes and must never name any. Output only the expanded query, with no explanations or preamble.

HUMAN PROMPT
------------------------------------------------------------
A user provides a plain-English legal query related to marriage in India.
Rewrite it using formal legal phrasing and add closely related legal terms, synonyms, and procedural keywords.

STRICT RULES:
- Preserve the original intent exactly — do NOT introduce new legal concepts not present in the query.
- Do NOT name, cite, or introduce any specific statute, Act, rule, section number, year,
  case name, court, or legal authority UNLESS it appears verbatim in the user's question.
  (e.g. if the user names only the "Hindu Marriage Act", do NOT add the "Anand Marriage Act",
  "Special Marriage Act", sectio

In [3]:
n = len(df)
for i, row in df.iterrows():
    key = str(i)
    if key in cache:
        print(f'⏩ [{i + 1}/{n}] cached, skipping')
        continue

    cache[key] = expand_query(row['Question'])
    ckpt.write_text(json.dumps(cache))   # persist after every call

    print(f'✅ [{i + 1}/{n}]')
    print(f'   Original : {row["Question"]}')
    print(f'   Expanded : {cache[key]}\n')
    time.sleep(0.5)                       # gentle on rate limits

df['Expanded_Query'] = [cache[str(i)] for i in range(n)]
df.to_csv(config.BENCHMARK_EXPANDED_CSV, index=False)
ckpt.unlink(missing_ok=True)
print('Saved', config.BENCHMARK_EXPANDED_CSV.relative_to(config.ROOT_DIR))

✅ [1/15]
   Original : I am a Sikh man planning to get married in Punjab. Does my marriage need to be registered under the Hindu Marriage Act?
   Expanded : What are the legal requirements and procedural obligations for registration and solemnization of a Sikh marriage in Punjab, and is such a marriage mandatorily registrable under the Hindu Marriage Act? What documentary requirements, jurisdictional conditions, and eligibility criteria apply to the registration process before the competent authority? Does the Hindu Marriage Act govern the formal registration of marriages solemnized according to Sikh rites and ceremonies, and what are the legal consequences of registration or non-registration of such a matrimonial union under the applicable personal law framework in Punjab?

✅ [2/15]
   Original : As a Muslim woman in Uttar Pradesh, if my husband pronounces 'Talaq' three times in one sitting, is my marriage legally terminated?
   Expanded : What is the legal validity and enforceability

### Results — original vs. expanded

Full input→output mapping plus how much richer the expanded queries are.

In [4]:
from IPython.display import display

orig_len = df['Question'].str.split().str.len()
exp_len = df['Expanded_Query'].str.split().str.len()
print(f'Avg length: {orig_len.mean():.0f} → {exp_len.mean():.0f} words '
      f'({exp_len.mean() / orig_len.mean():.1f}× richer)')

pd.set_option('display.max_colwidth', None)
display(df[['Question', 'Expanded_Query']])

Avg length: 23 → 84 words (3.7× richer)


,Question,Expanded_Query
0,I am a Sikh man planning to get married in Punjab. Does my marriage need to be registered under the Hindu Marriage Act?,"What are the legal requirements and procedural obligations for registration and solemnization of a Sikh marriage in Punjab, and is such a marriage mandatorily registrable under the Hindu Marriage Act? What documentary requirements, jurisdictional conditions, and eligibility criteria apply to the registration process before the competent authority? Does the Hindu Marriage Act govern the formal registration of marriages solemnized according to Sikh rites and ceremonies, and what are the legal consequences of registration or non-registration of such a matrimonial union under the applicable personal law framework in Punjab?"
1,"As a Muslim woman in Uttar Pradesh, if my husband pronounces 'Talaq' three times in one sitting, is my marriage legally terminated?","What is the legal validity and enforceability of instantaneous triple verbal pronouncement of divorce by a husband in a single sitting, and does such pronouncement constitute lawful dissolution of matrimonial union for a Muslim woman domiciled in Uttar Pradesh? What is the current legal status, judicial recognition, and procedural effect of such a unilateral oral declaration of marital termination, including its enforceability before competent authorities, and what are the legal consequences, rights, and remedies available to the wife following such a pronouncement of matrimonial dissolution?"
2,I am a Hindu woman from Kerala and my partner is a Christian man. Do we have to convert to marry each other legally?,"What are the legal requirements and procedural prerequisites for solemnization of an interfaith marriage between parties belonging to different religious denominations in India, specifically whether religious conversion is mandatory or obligatory for either spouse prior to or as a condition of valid matrimonial union? What documentary requirements, eligibility criteria, and formal procedures apply to the registration and legal recognition of such an inter-religious matrimonial alliance before a competent authority, and can such a marriage be solemnized and registered without either party undergoing religious conversion?"
3,"I belong to the Khasi tribe in Meghalaya. If I divorce my husband, who gets the custody of our children according to our customary laws?","In a matrimonial dissolution or divorce proceeding under recognized tribal customary law and indigenous personal law applicable to the Khasi community in Meghalaya, what are the governing principles and procedural norms for determining child custody, guardianship, and parental rights? How does customary law adjudicate custodial arrangements, welfare of the minor children, and parental responsibilities upon separation or dissolution of marriage within the Khasi tribal community, and what role does the competent authority or customary adjudicatory body play in resolving such custodial disputes?"
4,I am a Parsi man residing in Mumbai. Can I file for divorce in a regular family court?,"What is the jurisdictional competence of a regular family court to adjudicate dissolution of marriage proceedings initiated by a Parsi male domiciled in Mumbai? Does a standard family court possess the requisite authority to entertain a matrimonial petition for divorce filed by a member of the Parsi community, or must such proceedings be instituted before a specialized or designated tribunal? What are the procedural requirements, venue considerations, and forum selection criteria governing the filing of a divorce petition by a Parsi individual within the applicable territorial jurisdiction?"
5,"We are a Hindu couple residing in Goa. If we divorce, how is our property divided?","What are the legal provisions governing matrimonial property division and asset distribution upon dissolution of marriage or divorce for a Hindu couple domiciled or residing in Goa? What procedural requirements, enti